# Módulo 9 — Análise Espectral
## Seção 9.4 — Periodograma

**Objetivos.** Definir o periodograma e sua relação com a DFT; deduzir a forma em autocovariâncias amostrais; estudar a esperança como média espectral com janela de Fejér; compreender por que o periodograma não suavizado não é consistente; observar vazamento espectral e variabilidade por simulações reproduzíveis.

**Pré-requisitos.** Seções 8.6–8.8 (DFT, FFT, Nyquist) e 9.2–9.3 (densidade espectral e autocovariância).

**Fonte principal conferida.** Morettin e Toloi (2018), *Análise de Séries Temporais: Modelos Lineares Univariados*, v. 1, 3ª ed., **§14.4.1, pp. 367–369** (transformada finita e normalização) e **§14.4.2, pp. 369–370**, eqs. **(14.22)–(14.28)** (periodograma, distribuição assintótica e momentos). Os desenvolvimentos da janela de Fejér, as simulações e os comentários sobre vazamento são complementos matemático-computacionais coerentes com a teoria, não transcrições do livro.

---

### 1. O problema: estimar um espectro com dados finitos

A densidade espectral $f(\lambda)$ é definida para um processo populacional estacionário. Em pesquisa, dispomos apenas de $x_0,\ldots,x_{N-1}$. O **periodograma** é uma estimativa direta e simples da distribuição de potência por frequência, baseada no módulo quadrado da transformada discreta de Fourier.

Ele identifica oscilações e pode sugerir frequências dominantes, mas seus picos não devem ser interpretados isoladamente como significativos: uma série de ruído branco também apresenta picos aleatórios em uma amostra finita.

### 2. Definição: DFT e unidades

Para simplificar, suponha a média populacional $\mu$ conhecida e defina $z_t=x_t-\mu$. Em aplicações, é usual utilizar $z_t=x_t-\bar x$, o que modifica certos resultados exatos para amostras finitas.

Adotamos a convenção do `numpy.fft.fft`:

$$\boxed{D_N(\lambda)=\sum_{t=0}^{N-1}z_t e^{-i\lambda t}},\quad \lambda_k=\frac{2\pi k}{N},\ k=0,\ldots,N-1.\tag{9.4.1}$$

O **periodograma bilateral por radiano/amostra**, na mesma normalização espectral de Morettin e Toloi, é

$$\boxed{I_N(\lambda)=\frac{1}{2\pi N}|D_N(\lambda)|^2.}\tag{9.4.2}$$

**Comparação de convenções.** O livro emprega uma transformada finita normalizada por $\sqrt{2\pi N}$, eq. (14.18), e define o periodograma como seu módulo quadrado, eq. (14.22). Isso dá exatamente (9.4.2). A DFT bruta do NumPy não inclui essa normalização; devemos aplicá-la explicitamente ao calcular $I_N$.

Em ciclos por amostra $\nu=\lambda/(2\pi)$, a densidade de potência por ciclo é $P_N(\nu)=2\pi I_N(2\pi\nu)=|D_N|^2/N$. Para dados com intervalo físico $\Delta t$, convertemos para ciclos por unidade de tempo por mudança de variável. A troca de unidade altera as alturas das densidades, não a variância total.

### 3. Desenvolvimento algébrico da expressão em autocovariâncias

Expanda o quadrado:

$$\begin{aligned}
|D_N(\lambda)|^2
&=\left(\sum_{t=0}^{N-1}z_te^{-i\lambda t}\right)\left(\sum_{s=0}^{N-1}z_se^{i\lambda s}\right)\\
&=\sum_{t=0}^{N-1}\sum_{s=0}^{N-1}z_tz_s e^{-i\lambda(t-s)}.
\end{aligned}$$

Agrupamos os pares por atraso $h=t-s$. Para $h\ge0$, são $N-h$ pares. Defina a **autocovariância amostral com divisor $N$**:

$$\widehat\gamma_N(h)=\frac1N\sum_{t=0}^{N-1-h}z_{t+h}z_t,\quad 0\le h<N;\qquad\widehat\gamma_N(-h)=\widehat\gamma_N(h).$$

Então a igualdade algébrica exata (para qualquer sequência real $z_t$) é

$$\boxed{I_N(\lambda)=\frac1{2\pi}\sum_{h=-(N-1)}^{N-1}\widehat\gamma_N(h)e^{-i\lambda h}=\frac1{2\pi}\left[\widehat\gamma_N(0)+2\sum_{h=1}^{N-1}\widehat\gamma_N(h)\cos(h\lambda)\right].}\tag{9.4.3}$$

**Por que o divisor $N$ importa?** Se substituirmos $\widehat\gamma_N(h)$ pela versão com divisor $N-h$, a identidade (9.4.3) deixa de valer sem pesos adicionais. Esta é uma das razões para explicitar sempre a convenção de autocovariância amostral.

### 4. Identidade discreta de energia (Parseval)

Para $\lambda_k=2\pi k/N$, a DFT satisfaz

$$\sum_{t=0}^{N-1}|z_t|^2=\frac1N\sum_{k=0}^{N-1}|D_N(\lambda_k)|^2.$$

Multiplicando por $1/N$ e usando $\Delta\lambda=2\pi/N$,

$$\boxed{\underbrace{\frac1N\sum_{t=0}^{N-1}|z_t|^2}_{\widehat\gamma_N(0)}=\underbrace{\sum_{k=0}^{N-1}I_N(\lambda_k)\frac{2\pi}{N}}_{\text{área espectral discreta}}.}\tag{9.4.4}$$

Se $z_t=x_t-\bar x$, o primeiro membro é a variância amostral com divisor $N$, não a variância não viesada com divisor $N-1$.

### 5. Esperança do periodograma: efeito da janela finita

Agora suponha $X_t$ fracamente estacionário, com média conhecida e espectro contínuo $f$, conforme Seção 9.2. Então $E[z_tz_s]=\gamma(t-s)$. Tomando esperança da expressão dupla:

$$\begin{aligned}
E[I_N(\lambda)]
&=\frac1{2\pi N}\sum_{t,s=0}^{N-1}\gamma(t-s)e^{-i\lambda(t-s)}\\
&=\frac1{2\pi}\sum_{h=-(N-1)}^{N-1}\left(1-\frac{|h|}{N}\right)\gamma(h)e^{-i\lambda h}.\tag{9.4.5}
\end{aligned}$$

O fator $1-|h|/N$ aparece porque existem $N-|h|$ pares de observações separados por $h$. É a **janela triangular de Bartlett no domínio dos atrasos**.

Pela relação inversa $\gamma(h)=\int_{-\pi}^{\pi}f(\omega)e^{ih\omega}\,d\omega$, substituímos em (9.4.5) e trocamos soma finita e integral:

$$E[I_N(\lambda)]=\int_{-\pi}^{\pi} f(\omega)K_N(\lambda-\omega)\,d\omega,\tag{9.4.6}$$

onde a **janela (núcleo) de Fejér** é

$$\boxed{K_N(\theta)=\frac{1}{2\pi N}\left|\sum_{t=0}^{N-1}e^{-it\theta}\right|^2=\frac1{2\pi N}\left(\frac{\sin(N\theta/2)}{\sin(\theta/2)}\right)^2,}\tag{9.4.7}$$

com valor no ponto $\theta\equiv0\pmod{2\pi}$ definido pelo limite $N/(2\pi)$. Note que $K_N\ge0$ e $\int_{-\pi}^{\pi}K_N(\theta)\,d\theta=1$.

**Interpretação:** a esperança do periodograma é uma média local do espectro verdadeiro, não necessariamente $f(\lambda)$ em amostras finitas. Sob continuidade e condições usuais, $K_N$ concentra massa próximo de zero quando $N$ cresce, e então $E[I_N(\lambda)]\to f(\lambda)$ nos pontos adequados. Isso é uma forma de **assintótica ausência de viés**, diferente de consistência.

### 6. Por que o periodograma bruto não é consistente?

Morettin e Toloi, §14.4.2, pp. 369–370, discutem a dispersão assintótica das ordenadas do periodograma. Sob condições usuais de regularidade, para frequência interior fixa com $f(\lambda)>0$, a razão $I_N(\lambda)/f(\lambda)$ tem comportamento assintótico associado a uma variável exponencial de média 1, de modo que sua variância não tende a zero.

Um exemplo especialmente claro é um **ruído branco gaussiano** de variância $\sigma^2$. Nas frequências de Fourier interiores, $k\ne0$ e, quando $N$ par, $k\ne N/2$,

$$\boxed{\frac{I_N(\lambda_k)}{f(\lambda_k)}\sim \operatorname{Exp}(1),\qquad f=\frac{\sigma^2}{2\pi},}$$

para amostras gaussianas com média populacional conhecida (exatidão da distribuição nas frequências interiores). Assim,

$$E[I_N(\lambda_k)]=f,\qquad \operatorname{Var}(I_N(\lambda_k))=f^2.$$

Mesmo com $N$ grande, as ordenadas isoladas não se concentram ao redor de $f$. **Conclusão:** o periodograma bruto não é um estimador consistente do espectro. Para reduzir variabilidade, estudaremos suavização e Welch nas Seções 9.6–9.7.

**Cuidado:** nos pontos $k=0$ e $k=N/2$ (quando existente), o comportamento é diferente porque os coeficientes da DFT são reais; não aplicar a fórmula exponencial a esses casos.

### 7. Frequências dominantes, vazamento e zero-padding

Um sinal $z_t=A\cos(2\pi\nu_0t)$ com $\nu_0=k_0/N$ coincide com a grade de Fourier; se a amostra compreende um número inteiro de períodos e não há ruído, a potência fica concentrada nos bins correspondentes. Se $\nu_0$ não pertence à grade, a janela retangular de observação provoca **vazamento espectral**: o periodograma distribui energia por bins vizinhos e mais distantes.

- **Resolução da grade:** $\Delta\nu=1/N$ ciclos/amostra, sem preenchimento.
- **Zero-padding:** acrescentar zeros antes da FFT interpola a amostragem em frequência, mas não amplia a duração observada nem melhora, por si só, a resolução física entre componentes próximas.
- **Sinal, ruído e inferência:** o maior pico não constitui, sozinho, prova de periodicidade ou significância estatística.

### 8. Experimento 1 — Periodograma e conservação de energia

Vamos calcular a DFT com NumPy, normalizar pela convenção do livro e verificar (9.4.4). A função `np.fft.fftfreq` fornece frequências em ciclos por amostra; para obter rad/amostra, multiplicamos por $2\pi$.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(9401)
N = 256
t = np.arange(N)
x = 1.4*np.cos(2*np.pi*0.125*t) + 0.6*np.cos(2*np.pi*0.28*t) + rng.normal(0,.45,N)
z = x - x.mean()
X = np.fft.fft(z)
I = np.abs(X)**2/(2*np.pi*N)
nu = np.fft.fftfreq(N, d=1.0)

variancia_tempo = np.mean(z**2)
variancia_freq = np.sum(I)*(2*np.pi/N)
print('Variância temporal:', variancia_tempo)
print('Área espectral da DFT:', variancia_freq)
assert np.isclose(variancia_tempo, variancia_freq, atol=1e-10)

mask = (nu >= 0) & (nu <= 0.5)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(nu[mask], 2*np.pi*I[mask], lw=1)
ax.set(xlabel='Frequência (ciclos/amostra)',ylabel='Periodograma bilateral por ciclo',title='Periodograma bruto de duas oscilações com ruído')
ax.grid(alpha=.2); plt.show()

Variância temporal: 1.3670322771746368
Área espectral da DFT: 1.367032277174637


/tmp/ipykernel_2974/3978478654.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.grid(alpha=.2); plt.show()


A curva exibida é bilateral por ciclo, restrita a frequências não negativas para visualização. Por isso, **não** devemos interpretar diretamente a área sob apenas essa metade como a variância completa sem considerar o lado negativo. Para uma apresentação unilateral, as componentes interiores precisam ser duplicadas de maneira consistente.

### 9. Experimento 2 — Alta variabilidade do periodograma do ruído branco

Simularemos várias séries gaussianas e compararemos a ordenada do periodograma para um mesmo bin interior. O valor teórico é constante, mas as ordenadas variam intensamente entre amostras, mesmo quando $N$ cresce.

In [2]:
rng = np.random.default_rng(9402)
sigma2 = 1.0
f_real = sigma2/(2*np.pi)

for N in [128, 512, 2048]:
    repeticoes = 700
    k = N//8   # frequência interior fixa: lambda=pi/4
    amostras = rng.normal(0, np.sqrt(sigma2), size=(repeticoes,N))
    coef = np.fft.fft(amostras, axis=1)[:,k]
    valores = np.abs(coef)**2/(2*np.pi*N)
    print(f'N={N:4d}: média={valores.mean():.4f}; variância={valores.var():.4f}; esperadas={f_real:.4f}, {f_real**2:.4f}')
    assert abs(valores.mean()-f_real) < 0.035

fig, ax = plt.subplots(figsize=(8,4))
ax.hist(valores/f_real, bins=35, density=True, alpha=.65, label='Valores simulados / f')
u = np.linspace(0,7,400)
ax.plot(u,np.exp(-u),label='Densidade Exp(1)')
ax.set(xlabel='Ordenada normalizada',ylabel='Densidade',title='Ruído branco: flutuação do periodograma bruto')
ax.legend(); ax.grid(alpha=.2); plt.show()

N= 128: média=0.1590; variância=0.0239; esperadas=0.1592, 0.0253
N= 512: média=0.1679; variância=0.0260; esperadas=0.1592, 0.0253
N=2048: média=0.1535; variância=0.0209; esperadas=0.1592, 0.0253


/tmp/ipykernel_2974/3560074571.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); ax.grid(alpha=.2); plt.show()


### 10. Experimento 3 — Vazamento e zero-padding

In [3]:
N = 128
t = np.arange(N)
fig, ax = plt.subplots(figsize=(8,4))
for freq, estilo in [(12/N,'-'), (12.35/N,'--')]:
    sinal = np.cos(2*np.pi*freq*t)
    P = np.abs(np.fft.rfft(sinal))**2/N
    ax.plot(np.fft.rfftfreq(N),P,estilo,label=f'freq. real={freq:.5f} ciclos/amostra')
ax.set_xlim(.04,.16)
ax.set(xlabel='Frequência (ciclos/amostra)',ylabel='|DFT|²/N',title='Frequência na grade vs. fora da grade')
ax.legend(); ax.grid(alpha=.2); plt.show()

freq_real = 12.35/N
sinal = np.cos(2*np.pi*freq_real*t)
N_pad = 8*N
f_pad = np.fft.rfftfreq(N_pad)
P_pad = np.abs(np.fft.rfft(sinal, n=N_pad))**2/N
print('Passo da grade sem padding:', 1/N)
print('Passo da grade com padding:', 1/N_pad)
print('Duração observada em ambos os casos:', N, 'amostras')
assert 1/N_pad < 1/N

Passo da grade sem padding: 0.0078125
Passo da grade com padding: 0.0009765625
Duração observada em ambos os casos: 128 amostras


/tmp/ipykernel_2974/2855167965.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); ax.grid(alpha=.2); plt.show()


### 11. Limitações e cuidados metodológicos

1. É necessário justificar estacionariedade ou transformar/modelar tendências antes da interpretação de um espectro estacionário.
2. Periodogramas apresentam **variância elevada**; picos aparentes podem ser causados pelo acaso.
3. Tendência, amostragem inadequada, aliasing e janela de observação podem distorcer a interpretação.
4. Detrending, remoção da média, escolha da janela e normalização precisam ser documentados na pesquisa.
5. Periodogramas de séries não gaussianas ou com dependência intensa exigem cuidado adicional nas aproximações probabilísticas.

### 12. Exercícios para desenvolver autonomia

1. Parta de $|D_N(\lambda)|^2$ e obtenha (9.4.3) sem consultar a demonstração; explique por que surge o fator $N-|h|$ na esperança.
2. Para $z=(1,-1,1,-1)$, calcule a DFT à mão e o periodograma nas quatro frequências. Confira Parseval.
3. Crie pseudocódigo que calcule o periodograma *sem* `np.fft`, usando apenas exponenciais complexas e uma soma dupla; teste contra a FFT.
4. Simule 300 séries gaussianas de tamanhos 128 e 1024. Compare a dispersão da mesma ordenada normalizada do periodograma; interprete à luz da consistência.
5. Formule, com suas palavras, por que preencher com zeros não cria novas observações e não resolve duas frequências muito próximas.

**Síntese.** O periodograma é a porta de entrada para estimar o espectro, mas não é uma estimativa estável isoladamente. A próxima seção trata da **identificação de frequências dominantes** e das precauções necessárias para separar sinal, ruído e artefatos de estimação.